# 3 Rephrase

This notebook builds the rephrased branch for the study.

It does four things:
1. Rephrase AI proposals for each requested condition.
2. Rephrase AI reviews for each requested condition.
3. Rephrase shared human proposals once.
4. Rephrase shared human reviews once.

It does not generate new proposals or reviews, prepare embeddings, or run analyses.

In [1]:
PROPOSAL_CONDITIONS_TO_REPHRASE = ['one_at_a_time', 'persona']
REVIEW_CONDITIONS_TO_REPHRASE = ['baseline', 'one_at_a_time', 'persona']

REPHRASE_MODEL = 'gemini-3.1-pro-preview'
REPHRASE_TEMPERATURE = 0
MAX_TOKENS_PROPOSALS = 12000
MAX_TOKENS_REVIEWS = 4000
RETRY_DELAYS = [2, 5, 10]
SAVE_PROGRESS_EVERY_N_ROWS = 5
RESUME_OK = True

# Set to a fixed string only if you want to pin new outputs to a custom run id.
RUN_ID = None


In [2]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ai_models_interface import AIModelsInterface
from rephrase_pipeline import (
    build_rephrase_job_registry,
    find_project_root,
    load_human_proposal_sources,
    load_human_review_sources,
    locate_latest_ai_proposal_files,
    locate_latest_ai_review_files,
    now_run_id,
    rephrase_ai_proposals_for_condition,
    rephrase_ai_reviews_for_condition,
    rephrase_shared_human_proposals,
    rephrase_shared_human_reviews,
)

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
ai_interface = AIModelsInterface(config_path=str(PROJECT_ROOT / '.env'), override_env=True)
available_models = ai_interface.get_available_models()
resolved_model = ai_interface.resolve_model_name(REPHRASE_MODEL)
if resolved_model not in available_models:
    raise RuntimeError(f'Rephrase model unavailable with current API keys: {resolved_model}')
stage_run_id = RUN_ID or now_run_id()

# --- Registry 1: everything whose sources are ready now ---
# AI proposals + both human families. This registry never touches AI review
# files, so it is NOT blocked while review generation is still running. Run this
# cell and the AI-proposal / human rephrase cells now.
ai_proposal_sources = locate_latest_ai_proposal_files(PROJECT_ROOT, PROPOSAL_CONDITIONS_TO_REPHRASE)
human_proposal_sources = load_human_proposal_sources(PROJECT_ROOT)
human_review_sources = load_human_review_sources(PROJECT_ROOT)

proposal_registry = build_rephrase_job_registry(
    ai_proposal_sources=ai_proposal_sources,
    ai_review_sources={},
    human_proposal_sources=human_proposal_sources,
    human_review_sources=human_review_sources,
    rephrase_model=resolved_model,
    rephrase_temperature=REPHRASE_TEMPERATURE,
    run_id=stage_run_id,
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Rephrase model: {resolved_model}')
print(f'AI proposal jobs:    {len(proposal_registry["ai_proposal_rephrase_jobs"])} ready={sorted(ai_proposal_sources)}')
print(f'Human proposal jobs: {len(proposal_registry["human_proposal_rephrase_jobs"])}')
print(f'Human review jobs:   {len(proposal_registry["human_review_rephrase_jobs"])}')

INFO:ai_models_interface:OpenAI GPT-5.5 initialized
INFO:ai_models_interface:Google Gemini initialized
INFO:ai_models_interface:Anthropic Claude initialized


Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal
Rephrase model: gemini-3.1-pro-preview
AI proposal jobs:    2 ready=['one_at_a_time', 'persona']
Human proposal jobs: 2
Human review jobs:   2


In [3]:
ai_proposal_rephrase_outputs = {}

for job in proposal_registry['ai_proposal_rephrase_jobs']:
    condition = job['condition']
    print(f'\n=== Rephrase AI proposals: {condition} ===')
    result = rephrase_ai_proposals_for_condition(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        condition=condition,
        source_path=job['source_path'],
        rephrase_model=resolved_model,
        rephrase_temperature=REPHRASE_TEMPERATURE,
        max_tokens=MAX_TOKENS_PROPOSALS,
        retry_delays=RETRY_DELAYS,
        save_every_n_rows=SAVE_PROGRESS_EVERY_N_ROWS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    ai_proposal_rephrase_outputs[condition] = result
    print(f"Source file: {job['source_path']}")
    print(f"Output file: {result['output_path']}")
    print(f"Rows: {len(result['rephrased_df'])}")
    if result['qa_issues']:
        print('QA issues:')
        for issue in result['qa_issues'][:10]:
            print(f'  - {issue}')


=== Rephrase AI proposals: one_at_a_time ===
Source file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/ai-proposals/one_at_a_time/ai_proposals_one_at_a_time_complete_20260709_095141.csv
Output file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/ai-proposals/one_at_a_time/rephrased/ai_proposals_one_at_a_time_rephrased_20260709_162605.csv
Rows: 69

=== Rephrase AI proposals: persona ===
Source file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/ai-proposals/persona/ai_proposals_persona_complete_20260709_095141.csv
Output file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/ai-proposals/persona/rephrased/ai_proposals_persona_rephrased_20260709_162605.csv
Rows: 69


In [4]:
# --- Registry 2: AI reviews (run once review generation is complete) ---
# require_all=False builds jobs for whichever review conditions already have a
# complete file and skips the rest, so you can rephrase reviews incrementally.
# Re-run this cell (and the AI-review rephrase cell below) as more conditions
# finish. Running it now with no reviews ready simply yields 0 jobs.
ai_review_sources = locate_latest_ai_review_files(
    PROJECT_ROOT, REVIEW_CONDITIONS_TO_REPHRASE, require_all=False
)
pending_reviews = [c for c in REVIEW_CONDITIONS_TO_REPHRASE if c not in ai_review_sources]

review_registry = build_rephrase_job_registry(
    ai_proposal_sources={},
    ai_review_sources=ai_review_sources,
    human_proposal_sources={},
    human_review_sources={},
    rephrase_model=resolved_model,
    rephrase_temperature=REPHRASE_TEMPERATURE,
    run_id=stage_run_id,
)

print(f'AI review jobs: {len(review_registry["ai_review_rephrase_jobs"])} ready={sorted(ai_review_sources)}')
if pending_reviews:
    print(f'Pending (not yet complete): {pending_reviews}')
    print('Re-run this cell + the AI-review rephrase cell below after those finish generating.')

AI review jobs: 3 ready=['baseline', 'one_at_a_time', 'persona']


In [5]:
ai_review_rephrase_outputs = {}

for job in review_registry['ai_review_rephrase_jobs']:
    condition = job['condition']
    print(f'\n=== Rephrase AI reviews: {condition} ===')
    result = rephrase_ai_reviews_for_condition(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        condition=condition,
        source_path=job['source_path'],
        rephrase_model=resolved_model,
        rephrase_temperature=REPHRASE_TEMPERATURE,
        max_tokens=MAX_TOKENS_REVIEWS,
        retry_delays=RETRY_DELAYS,
        save_every_n_rows=SAVE_PROGRESS_EVERY_N_ROWS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    ai_review_rephrase_outputs[condition] = result
    print(f"Source file: {job['source_path']}")
    print(f"Output file: {result['output_path']}")
    print(f"Rows: {len(result['rephrased_df'])}")
    if result['qa_issues']:
        print('QA issues:')
        for issue in result['qa_issues'][:10]:
            print(f'  - {issue}')

INFO:root:AFC is enabled with max remote calls: 10.



=== Rephrase AI reviews: baseline ===


INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote 

KeyboardInterrupt: 

In [8]:
human_proposal_rephrase_outputs = {}
human_review_rephrase_outputs = {}

for job in proposal_registry['human_proposal_rephrase_jobs']:
    cohort = job['cohort']
    print(f'\n=== Rephrase shared human proposals: {cohort} ===')
    result = rephrase_shared_human_proposals(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        source_path=job['source_path'],
        cohort=cohort,
        rephrase_model=resolved_model,
        rephrase_temperature=REPHRASE_TEMPERATURE,
        max_tokens=MAX_TOKENS_PROPOSALS,
        retry_delays=RETRY_DELAYS,
        save_every_n_rows=SAVE_PROGRESS_EVERY_N_ROWS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    human_proposal_rephrase_outputs[cohort] = result
    print(f"CSV file: {result['csv_path']}")
    print(f"JSON file: {result['json_path']}")
    print(f"Rows: {len(result['rephrased_df'])}")

for job in proposal_registry['human_review_rephrase_jobs']:
    cohort = job['cohort']
    print(f'\n=== Rephrase shared human reviews: {cohort} ===')
    result = rephrase_shared_human_reviews(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        source_path=job['source_path'],
        cohort=cohort,
        rephrase_model=resolved_model,
        rephrase_temperature=REPHRASE_TEMPERATURE,
        max_tokens=MAX_TOKENS_REVIEWS,
        retry_delays=RETRY_DELAYS,
        save_every_n_rows=SAVE_PROGRESS_EVERY_N_ROWS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    human_review_rephrase_outputs[cohort] = result
    print(f"CSV file: {result['csv_path']}")
    print(f"Rows: {len(result['rephrased_df'])}")
    if result['qa_issues']:
        print('QA issues:')
        for issue in result['qa_issues'][:10]:
            print(f'  - {issue}')

INFO:root:AFC is enabled with max remote calls: 10.



=== Rephrase shared human proposals: y1 ===
CSV file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/human-proposals/rephrased/human_proposals_rephrased_y1_20260709_162605.csv
JSON file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/human-proposals/rephrased/human_proposals_rephrased_y1_20260709_162605.json
Rows: 12

=== Rephrase shared human proposals: y2 ===
CSV file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/human-proposals/rephrased/human_proposals_rephrased_y2_20260709_162605.csv
JSON file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/human-proposals/rephrased/human_proposals_rephrased_y2_20260709_162605.json
Rows: 11

=== Rephrase shared human reviews: human-y1 ===
CSV file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/reviews/human_reviews/rephrased/human_reviews_human-y1_rephrased_20260709_162605.csv
Rows: 47

=== Rephrase shared human reviews: human-y2 ===


INFO:root:AFC remote call 1 is done.


CSV file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/reviews/human_reviews/rephrased/human_reviews_human-y2_rephrased_20260709_162605.csv
Rows: 38


In [9]:
summary_rows = []

for condition, result in ai_proposal_rephrase_outputs.items():
    summary_rows.append({
        'artifact_family': 'ai_proposals',
        'scope': condition,
        'rows': len(result['rephrased_df']),
        'output_file': str(result['output_path']),
        'reused_existing': result['reused_existing'],
        'qa_issue_count': len(result['qa_issues']),
    })

for condition, result in ai_review_rephrase_outputs.items():
    summary_rows.append({
        'artifact_family': 'ai_reviews',
        'scope': condition,
        'rows': len(result['rephrased_df']),
        'output_file': str(result['output_path']),
        'reused_existing': result['reused_existing'],
        'qa_issue_count': len(result['qa_issues']),
    })

for cohort, result in human_proposal_rephrase_outputs.items():
    summary_rows.append({
        'artifact_family': 'human_proposals',
        'scope': cohort,
        'rows': len(result['rephrased_df']),
        'output_file': str(result['csv_path']),
        'reused_existing': result['reused_existing'],
        'qa_issue_count': len(result['qa_issues']),
    })

for cohort, result in human_review_rephrase_outputs.items():
    summary_rows.append({
        'artifact_family': 'human_reviews',
        'scope': cohort,
        'rows': len(result['rephrased_df']),
        'output_file': str(result['csv_path']),
        'reused_existing': result['reused_existing'],
        'qa_issue_count': len(result['qa_issues']),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


,artifact_family,scope,rows,output_file,reused_existing,qa_issue_count
0,ai_proposals,one_at_a_time,69,/Users/eveyhuang/Documents/NICO/human-AI-propo...,True,0
1,ai_proposals,persona,69,/Users/eveyhuang/Documents/NICO/human-AI-propo...,True,0
2,human_proposals,y1,12,/Users/eveyhuang/Documents/NICO/human-AI-propo...,True,0
3,human_proposals,y2,11,/Users/eveyhuang/Documents/NICO/human-AI-propo...,True,0
4,human_reviews,human-y1,47,/Users/eveyhuang/Documents/NICO/human-AI-propo...,True,0
5,human_reviews,human-y2,38,/Users/eveyhuang/Documents/NICO/human-AI-propo...,False,0
